In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [24]:
# 考虑优化方向：增加层的dropout，全连接层变1*1卷积层
class MultiQueryAttention(nn.Module):
    def __init__(self, heads, dims, attn_scale=None, sr_scale=1):
        super().__init__()
        self.dims = dims
        self.heads = heads
        self.dim_per_head = dims / heads
        
        # 定义获得query、key、value的投影张量
        self.proj_q = nn.Linear(dims, dims * heads)
        self.proj_k = nn.Linear(dims, dims)
        self.proj_v = nn.Linear(dims, dims)
        
        # 考虑下采样
        self.sr_scale = sr_scale
        self.sr = sr_scale
        if sr_scale > 1:
            self.sr = nn.Conv2d(dims, dims, kernel_size=sr_scale, stride=sr_scale)
            self.norm = nn.LayerNorm(dims)
            
        # 考虑注意力分数缩放
        self.scale = attn_scale or self.dim_per_head ** -0.5
        
        # 定义最终投影层，学习多注意力头组合的形式
        self.proj = nn.Conv2d(heads * dims, dims, kernel_size=1, stride=1, bias=True)
        
    def forward(self, x):
        # 得到输入的尺寸(B, C, H, W)
        batch_size, dim, height, width = x.shape
    
        assert height * width == self.sr_scale ** 2 * dim, print("resolution is not compatible with dimension")
        
        # 得到query(B, h, N, C)
        query = self.proj_q(x.reshape(batch_size, -1, dim)).reshape(batch_size, self.heads, -1, dim)
        
        # x = SRA(x) (B, N/4, C)
        x = self.sr(x)
        x = self.norm(x.reshape(batch_size, -1, dim))
        
        # 得到key(B, 1, C, N/4)和value(B, 1, N/4, C)
        key = self.proj_k(x).reshape(batch_size, 1, dim, -1)
        value = self.proj_v(x).reshape(batch_size, 1, -1, dim)
        
        # Q、K相乘，经softmax函数得到注意力分数
        print(query.shape, key.shape)
        attn_socre = torch.matmul(query, key) * self.scale
        attn_score = F.softmax(attn_socre, dim=-1)

        # value加权，获得最终输出
        logits = torch.matmul(attn_socre, value).reshape(batch_size, dim * self.heads, height, width)
        print(logits.shape)
        x = self.proj(logits)
        
        return x

In [26]:
X = torch.ones(2, 16, 8, 8)
net = MultiQueryAttention(heads=4, dims=16, sr_scale=2)
net(X)

torch.Size([2, 4, 64, 16]) torch.Size([2, 1, 16, 16])
torch.Size([2, 64, 8, 8])


tensor([[[[ 0.1063,  0.0988,  0.0590,  ...,  0.0086,  0.1005,  0.0524],
          [ 0.0209, -0.0191,  0.0668,  ...,  0.0666,  0.0956,  0.0846],
          [-0.0055, -0.0009,  0.0238,  ...,  0.0549, -0.0019,  0.0279],
          ...,
          [ 0.0354,  0.0309,  0.0405,  ...,  0.0405,  0.0438,  0.0426],
          [ 0.0574,  0.0552,  0.0436,  ...,  0.0289,  0.0557,  0.0416],
          [ 0.0325,  0.0208,  0.0458,  ...,  0.0458,  0.0543,  0.0511]],

         [[ 0.0924,  0.0857,  0.0498,  ...,  0.0044,  0.0872,  0.0438],
          [ 0.0155, -0.0205,  0.0568,  ...,  0.0567,  0.0828,  0.0729],
          [-0.0083, -0.0041,  0.0180,  ...,  0.0461, -0.0051,  0.0217],
          ...,
          [ 0.0285,  0.0245,  0.0332,  ...,  0.0332,  0.0361,  0.0350],
          [ 0.0483,  0.0464,  0.0359,  ...,  0.0227,  0.0468,  0.0342],
          [ 0.0259,  0.0154,  0.0379,  ...,  0.0379,  0.0455,  0.0426]],

         [[ 0.0835,  0.0823,  0.0758,  ...,  0.0675,  0.0826,  0.0747],
          [ 0.0695,  0.0630,  